# 04 — GPU analysis (G1 + G3)

Throughput scaling vs batch size, and the 2×2 AMP × torch.compile grid, measured with CUDA events after warmup (compile time excluded).

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.train import build_model
from s2cd.gpu.scale import batch_scaling, amp_compile_grid

cfg = load_config("configs/train.yaml")
model = build_model(cfg).to(device).eval()
shape = (1, cfg["in_channels"], 256, 256)  # (n, c, h, w)
print("params:", model.count_params())


In [ ]:
scale_rows = batch_scaling(model, shape, device,
                          batch_sizes=(8, 16, 32, 64), iters=20)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot([r["batch"] for r in scale_rows],
         [r["samples_per_sec"] for r in scale_rows], "o-")
plt.xlabel("batch size"); plt.ylabel("samples / s"); plt.grid(True)
plt.title("Inference throughput vs batch size (AMP)")
plt.show()


In [ ]:
# G3 precision/perf grid — fresh model per cell (compile is one-shot)
grid_rows = amp_compile_grid(lambda: build_model(cfg), shape, device,
                             batch=32, chip_hw=256, iters=20)
for r in grid_rows:
    print(r)


In [ ]:
import json

rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
(rep / "gpu_scaling.json").write_text(json.dumps(
    {"batch_scaling": scale_rows, "amp_compile_grid": grid_rows,
     "device": describe_device(device)}, indent=1))
store.push_report("gpu_scaling.json")
print("pushed gpu_scaling.json -> Drive")
